# Step 4: LLaVA-1.5 on Kannada (a cautionary case), reported with tokenization numbers

LLaVA-1.5's language model (Vicuna / Llama-2) saw very little Kannada, and its 32k-token vocabulary has little Kannada coverage.
A low Kannada score here mostly means "cannot read Kannada". This notebook measures that, so the result can be reported honestly
and kept separate from the morphology question.

In [ ]:
# Shared setup for notebooks 02-04 (pinned transformers; do NOT run in the same session as notebook 01)
!pip install -q "transformers==4.57.1" "accelerate>=1.0" "huggingface_hub>=0.34"
import glob, os, sys
# The xling_spatial code: either a Kaggle dataset you uploaded (Add Input) or a git clone of your team repo.
REPO_URL = ""   # e.g. "https://github.com/<user>/<repo>.git" (leave empty if you attached the code as a Kaggle dataset)
def _find():
    return glob.glob("/kaggle/input/**/xling_spatial/__init__.py", recursive=True) + glob.glob("/kaggle/working/**/xling_spatial/__init__.py", recursive=True)
hits = _find()
if not hits and REPO_URL:
    !git clone -q {REPO_URL} /kaggle/working/repo
    hits = _find()
assert hits, "Set REPO_URL to your GitHub repo, or attach the 'xling-spatial-code' dataset (see README step 0)"
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)

import torch, transformers, numpy as np, pandas as pd, matplotlib.pyplot as plt
from xling_spatial import data, runner, analysis, attention
from xling_spatial.models import VLM
from xling_spatial.tokenizer_stats import tokenizer_table
print(torch.__version__, transformers.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

COCO_ROOT = data.find_coco_root(); print("COCO:", COCO_ROOT)
WORK = "/kaggle/working"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # needed only for gated models (Gemma 3)
except Exception:
    pass

def free(vlm):
    attention.uninstall()
    del vlm.model
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
PILOT_CSV = glob.glob("/kaggle/input/**/pilot*.csv", recursive=True)[0]
pilot = data.load_pilot(PILOT_CSV)
LANGS = [l for l in ("en", "hi", "kn") if l in data.languages(pilot)]
boxes = data.load_coco_boxes(COCO_ROOT, pilot.image.unique().tolist(), f"{WORK}/boxes_pilot.json")

In [ ]:
tok = transformers.AutoTokenizer.from_pretrained("llava-hf/llava-1.5-7b-hf")
qtok = transformers.AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
tt = tokenizer_table({"LLaVA-1.5 (Llama-2)": tok, "Qwen2.5-VL": qtok}, pilot, LANGS)
tt.to_csv(f"{WORK}/step4_tokenizers.csv", index=False); display(tt.round(3))
ex = pilot.dropna(subset=["text_kn"]).iloc[0].text_kn
print(ex); print(tok.tokenize(ex)); print(qtok.tokenize(ex))

In [ ]:
llava = VLM("llava-hf/llava-1.5-7b-hf", dtype="float16")
out = f"{WORK}/step4_llava15"
res = runner.run(llava, pilot, LANGS, COCO_ROOT, out, weights=(0.5, 1.0, 1.2), boxes=boxes, save_maps=True)
att = pd.read_csv(f"{out}/attention.csv", dtype={"item_id": str})

In [ ]:
s = analysis.summarize(res); s.to_csv(f"{WORK}/step4_llava15_summary.csv", index=False)
display(s[np.isclose(s.weight, 1.0)].round(3))
base = res[np.isclose(res.weight, 1.0)]
print("yes-rate by language (near 0 or 1 = the model ignores the statement):")
print(base.groupby("lang").pred.mean().round(3))
analysis.plot_layers(att, title="LLaVA-1.5 · last-token attention by language"); plt.savefig(f"{WORK}/step4_layers.png", dpi=150); plt.show()

**How to report it:** put LLaVA-1.5's accuracy, yes-rate and `yesno_mass` per language in one table next to
`tokens_per_word`, `byte_fallback_share` and `parity_vs_en`. If Kannada sits near chance with a heavy yes-bias and high byte-fallback,
say the model cannot read the script, and leave LLaVA-1.5 out of the morphology analysis (keep it as the AdaptVis-reproduction model only).